# Deep Past Initiative: Data Preprocessing

This notebook covers all data preprocessing steps for the Akkadian → English machine translation pipeline:
1. Setup & imports
2. Load competition and external datasets
3. Normalize transliterations and translations
4. Cross-dataset deduplication
5. Genre prefix conditioning
6. Stage dataset assembly
7. Glossary construction

## 1. Setup & Imports

In [ ]:
# pip install -r requirements.txt

  Using cached torch-2.10.0-cp314-cp314-win_amd64.whl.metadata (31 kB)
  Using cached torchvision-0.25.0-cp314-cp314-win_amd64.whl.metadata (5.4 kB)
  Using cached torchaudio-2.10.0-cp314-cp314-win_amd64.whl.metadata (6.7 kB)
  Using cached transformers-4.40.0-py3-none-any.whl.metadata (137 kB)
  Using cached sentencepiece-0.2.1-cp314-cp314-win_amd64.whl.metadata (10 kB)
  Using cached sacrebleu-2.6.0-py3-none-any.whl.metadata (39 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached datasets-4.7.0-py3-none-any.whl.metadata (19 kB)
  Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached sacremoses-0.1.1-py3-none-any.whl.metadata (8.3 kB)
  Using cached unicodedata2-17.0.1-cp314-cp314-win_amd64.whl.metadata (3.5 kB)
  Using c

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [24 lines of output]
      Checking for Rust toolchain....
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp314-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\tronl\AppData\Local\puccinialin\puccinialin\Cache
      Rustup already downloaded
      Installing rust to C:\Users\tronl\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: It looks like you have an existing rustup settings file at:
      warn: C:\Users\tronl\.rustup\settings.toml
      warn: Rustup will install the default toolchain as specified in the settings file,
      warn: instead of the one inferred from the default host triple.
      warn: installing msvc toolchain without its prerequisites
      info: profile set to 'minimal'
      info: default host triple is x86_64-pc-windows-ms

In [2]:
import os
import re
import json
import random
import unicodedata

import numpy as np
import pandas as pd

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Ensure output directories exist
os.makedirs('data', exist_ok=True)
os.makedirs('data/akkademia', exist_ok=True)
os.makedirs('models/stage1_final', exist_ok=True)
os.makedirs('models/stage2_final', exist_ok=True)
os.makedirs('models/stage3_final', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print('Setup complete.')

ImportError: 

IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!

Importing the numpy C-extensions failed. This error can happen for
many reasons, often due to issues with your setup or how NumPy was
installed.

We have compiled some common reasons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python 3.14 from "d:\Codes\SIT\Pbl\deep-past-challenge\.venv\Scripts\python.exe"
  * The NumPy version is: "2.4.3"

and make sure that they are the versions you expect.

Please carefully study the information and documentation linked above.
This is unlikely to be a NumPy issue but will be caused by a bad install
or environment on your machine.

Original error was: DLL load failed while importing _multiarray_umath: The specified procedure could not be found.


## 2. Load Datasets

### 2a. Competition Datasets

In [ ]:
train_df              = pd.read_csv('data/competition/train.csv')
test_df               = pd.read_csv('data/competition/test.csv')
sample_submission_df  = pd.read_csv('data/competition/sample_submission.csv')
sentences_oare_df     = pd.read_csv('data/competition/Sentences_Oare_FirstWord_LinNum.csv')
published_texts_df    = pd.read_csv('data/competition/published_texts.csv')
lexicon_df            = pd.read_csv('data/competition/OA_Lexicon_eBL.csv')
dictionary_df         = pd.read_csv('data/competition/eBL_Dictionary.csv')

print('Competition datasets loaded.')
print(f'  train:             {train_df.shape}')
print(f'  test:              {test_df.shape}')
print(f'  sample_submission: {sample_submission_df.shape}')
print(f'  sentences_oare:    {sentences_oare_df.shape}')
print(f'  published_texts:   {published_texts_df.shape}')
print(f'  lexicon:           {lexicon_df.shape}')
print(f'  dictionary:        {dictionary_df.shape}')
train_df.head(2)

### 2b. Akkademia Dataset

In [ ]:
def load_akkademia_dataset():
    """Load Akkademia NMT_input files and return aligned DataFrames per split."""
    akkademia_data = {}
    for split in ['train', 'valid', 'test']:
        with open(f'data/akkademia/{split}.ak', 'r', encoding='utf-8') as f:
            akkadian_lines = [line.strip() for line in f]
        with open(f'data/akkademia/{split}.en', 'r', encoding='utf-8') as f:
            english_lines = [line.strip() for line in f]
        akkademia_data[split] = pd.DataFrame({
            'transliteration': akkadian_lines,
            'translation': english_lines
        })
    print('Akkademia dataset loaded.')
    for split, df in akkademia_data.items():
        print(f'  {split}: {len(df)} pairs')
    return akkademia_data

akkademia_data = load_akkademia_dataset()
akkademia_data['train'].head(2)

### 2c. ORACC Kaggle Dataset

In [ ]:
oracc_df = pd.read_csv('data/oracc_kaggle/train.csv')
print(f'ORACC dataset loaded. Shape: {oracc_df.shape}')
print('Columns:', oracc_df.columns.tolist())

# Identify genre priority (used later for sample weighting)
GENRE_PRIORITY = {
    'HIGH':   ['letter', 'administrative', 'memo', 'note', 'decree', 'report'],
    'MEDIUM': ['legal', 'contract', 'list'],
    'LOW':    ['royal inscription', 'votive inscription', 'building inscription', 'annals'],
}
oracc_df.head(2)

## 3. Text Normalization

In [ ]:
# Unicode subscript digits → ASCII
_SUBSCRIPT_MAP = str.maketrans('₀₁₂₃₄₅₆₇₈₉', '0123456789')

def normalize_transliteration(text):
    """Normalize Akkadian transliteration text.

    Steps:
      - Unicode NFC normalization
      - Subscript digit mapping (₁ → 1, etc.)
      - Replace damage/lacuna markers [...] with <gap>
      - Strip scribal correction markers (!, ?, #)
      - Collapse whitespace
    """
    if pd.isna(text) or text == '':
        return text
    text = unicodedata.normalize('NFC', text)
    text = text.translate(_SUBSCRIPT_MAP)
    text = re.sub(r'\[.*?\]', '<gap>', text)
    text = re.sub(r'[!?#]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def normalize_translation(text):
    """Normalize English translation text.

    Steps:
      - Normalize fancy quotation marks to ASCII
      - Normalize whitespace
      - Strip leading/trailing editorial bracket marks
    """
    if pd.isna(text) or text == '':
        return text
    text = re.sub(r'[“”‟\u02ee\uff02]', '"', text)
    text = re.sub(r"[\u2018\u2019\u201b\u02bc\u055a]", "'", text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'^[\[\]()]+|[\[\]()]+$', '', text)
    return text


# --- Quick sanity checks ---
assert normalize_transliteration('a₁-šur [broken] text!') == 'a1-šur <gap> text'
assert normalize_translation('\u201cHello\u201d') == '"Hello"'
print('Normalization functions defined and verified.')

### 3a. Apply Normalization to All Datasets

In [ ]:
# Competition train
train_df['transliteration_normalized'] = train_df['transliteration'].apply(normalize_transliteration)
train_df['translation_normalized']     = train_df['translation'].apply(normalize_translation)

# Akkademia splits
for split in ['train', 'valid', 'test']:
    akkademia_data[split]['transliteration_normalized'] = (
        akkademia_data[split]['transliteration'].apply(normalize_transliteration)
    )
    akkademia_data[split]['translation_normalized'] = (
        akkademia_data[split]['translation'].apply(normalize_translation)
    )

# ORACC — columns are 'akkadian' and 'english'
oracc_df['transliteration_normalized'] = oracc_df['akkadian'].apply(normalize_transliteration)
oracc_df['translation_normalized']     = oracc_df['english'].apply(normalize_translation)

print('Normalization applied to all datasets.')
print('Competition train sample:')
train_df[['transliteration', 'transliteration_normalized']].head(2)

## 4. Cross-Dataset Deduplication

In [ ]:
# Build set of all normalized competition transliterations to exclude from external data
competition_translit_set = set(train_df['transliteration_normalized'].dropna())
print(f'Competition train unique transliterations: {len(competition_translit_set)}')

# --- Akkademia: remove rows that appear in competition train ---
akkademia_deduped = {}
for split in ['train', 'valid', 'test']:
    df = akkademia_data[split]
    before = len(df)
    df = df[~df['transliteration_normalized'].isin(competition_translit_set)].copy()
    df = df.drop_duplicates(subset=['transliteration_normalized'], keep='first')
    after = len(df)
    akkademia_deduped[split] = df
    print(f'  Akkademia {split}: {before} → {after} rows')

# --- ORACC: remove competition overlaps and internal duplicates ---
before = len(oracc_df)
oracc_deduped = oracc_df[
    ~oracc_df['transliteration_normalized'].isin(competition_translit_set)
].drop_duplicates(subset=['transliteration_normalized'], keep='first').copy()
print(f'  ORACC: {before} → {len(oracc_deduped)} rows')

# --- Sentences_Oare: exclude tablets that appear in competition train ---
before = len(sentences_oare_df)
sentences_oare_filtered = (
    sentences_oare_df[~sentences_oare_df['text_uuid'].isin(train_df['oare_id'])]
    .dropna(subset=['translation'])
    .copy()
)
print(f'  Sentences_Oare: {before} → {len(sentences_oare_filtered)} rows (nulls removed)')

## 5. Genre Prefix Conditioning

In [ ]:
GENRE_PREFIX_MAP = {
    'letter':         '[LETTER]',
    'debt note':      '[DEBT_NOTE]',
    'loan':           '[DEBT_NOTE]',
    'contract':       '[CONTRACT]',
    'administrative': '[ADMIN]',
    'admin':          '[ADMIN]',
}

def map_genre_to_prefix(genre_label):
    """Map a genre label string to its special prefix token."""
    if pd.isna(genre_label):
        return '[UNKNOWN]'
    g = genre_label.lower().strip()
    for key, prefix in GENRE_PREFIX_MAP.items():
        if key in g:
            return prefix
    return '[UNKNOWN]'


# Merge genre labels from published_texts into competition train
train_with_genre = train_df.merge(
    published_texts_df[['oare_id', 'genre_label']],
    on='oare_id',
    how='left'
)

train_with_genre['genre_prefix'] = train_with_genre['genre_label'].apply(map_genre_to_prefix)
train_with_genre['transliteration_prefixed'] = (
    train_with_genre['genre_prefix'] + ' ' + train_with_genre['transliteration_normalized']
)

print('Genre prefix conditioning applied.')
print('Genre distribution:')
print(train_with_genre['genre_prefix'].value_counts())
train_with_genre[['transliteration_normalized', 'genre_prefix', 'transliteration_prefixed']].head(3)

## 6. Stage Dataset Assembly

In [ ]:
# ── Stage 1: Akkademia (train + valid) + ORACC ──────────────────────────────
akkademia_stage1 = pd.concat([
    akkademia_deduped['train'],
    akkademia_deduped['valid'],
])

stage1_data = pd.concat([
    akkademia_stage1[['transliteration_normalized', 'translation_normalized']],
    oracc_deduped[['transliteration_normalized', 'translation_normalized']],
], ignore_index=True).rename(columns={
    'transliteration_normalized': 'transliteration',
    'translation_normalized':     'translation',
})
stage1_data.dropna(subset=['transliteration', 'translation'], inplace=True)
print(f'Stage 1 dataset: {len(stage1_data)} rows')

# ── Stage 2: Sentences_Oare rows not in competition train ───────────────────
stage2_data = sentences_oare_filtered[
    ['text_uuid', 'translation', 'first_word_spelling', 'line_number', 'side']
].copy()
print(f'Stage 2 dataset: {len(stage2_data)} rows')

# ── Stage 3: Competition train, 90/10 doc-order split ───────────────────────
stage3_full = train_with_genre[[
    'transliteration_prefixed', 'translation_normalized'
]].rename(columns={
    'transliteration_prefixed': 'transliteration',
    'translation_normalized':   'translation',
})

split_idx    = int(len(stage3_full) * 0.9)
stage3_train = stage3_full.iloc[:split_idx].copy()
stage3_val   = stage3_full.iloc[split_idx:].copy()
print(f'Stage 3 train: {len(stage3_train)} rows  |  val: {len(stage3_val)} rows')

# ── Save to disk ─────────────────────────────────────────────────────────────
stage1_data.to_csv('data/stage1_train.csv',  index=False)
stage2_data.to_csv('data/stage2_train.csv',  index=False)
stage3_train.to_csv('data/stage3_train.csv', index=False)
stage3_val.to_csv('data/stage3_val.csv',     index=False)

print('Stage datasets saved to data/.')

## 7. Glossary Construction

In [ ]:
# Extract personal names (PN), geographic names (GN), and month names (MN)
# from OA_Lexicon_eBL.csv to build a proper-name glossary.

names_df = lexicon_df[lexicon_df['type'].isin(['PN', 'GN', 'MN'])].copy()
glossary_dict = dict(zip(names_df['form'], names_df['norm']))

print(f'Glossary entries: {len(glossary_dict)}')
print('Sample entries:', dict(list(glossary_dict.items())[:5]))

with open('data/glossary.json', 'w', encoding='utf-8') as f:
    json.dump(glossary_dict, f, ensure_ascii=False, indent=2)

print('Glossary saved to data/glossary.json.')

## 8. Preprocessing Summary

In [ ]:
summary = {
    'Competition train rows':       len(train_df),
    'Competition test rows':        len(test_df),
    'Akkademia train (deduped)':    len(akkademia_deduped['train']),
    'Akkademia valid (deduped)':    len(akkademia_deduped['valid']),
    'Akkademia test  (deduped)':    len(akkademia_deduped['test']),
    'ORACC (deduped)':              len(oracc_deduped),
    'Sentences_Oare (filtered)':    len(sentences_oare_filtered),
    'Stage 1 train rows':           len(stage1_data),
    'Stage 2 train rows':           len(stage2_data),
    'Stage 3 train rows':           len(stage3_train),
    'Stage 3 val rows':             len(stage3_val),
    'Glossary entries':             len(glossary_dict),
}

summary_df = pd.DataFrame.from_dict(summary, orient='index', columns=['count'])
print(summary_df.to_string())